# Solar filaments: a reproducible instance-segmentation baseline

Predict each individual dark solar filament in a 2048 × 2048 H-alpha observation.
The ranking metric is **pooled per-annotator Panoptic Quality**, with strict IoU > 0.5.
This notebook is the canonical workflow for [solar-seg](https://github.com/srivatsav-kannan/solar-seg).
The modules contain the implementation; the notebook explains and runs the steps.

**Default run:** audit the official inputs, load the released checkpoint, inspect a calibration
example, run correctness checks, infer all test images on CPU, and validate the output CSV.
Training and holdout evaluation are explicit opt-ins because they are expensive and the
holdout has already been exposed during the recorded baseline campaign. A default inference
replay is not a claim that training has been rerun. No Kaggle submission occurs automatically.

Use only the competition's training JSON. The full public MAGFiLO labels overlap the test set.
No external images or pretrained filament models are used. Data retain their original terms.
See the repository's requirements register, literature review, validation protocol, and ledger.

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import urllib.request
import zipfile

SOURCE_COMMIT = '4cc77325af2e4bebba1de4f30f489db72ea35478'
EXPECTED_SOURCE = {'solarseg/__init__.py': '32ca1f85bbbc5f820eac7e237cd01ec16027cc5b3e15028d2af265483e0a9b5b', 'solarseg/__main__.py': 'f887e0af26ca51dc3e89f873e775ffdf4f55426a3d8d25204c4b065a8f312979', 'solarseg/data.py': '46152a04d6eafbbaeaa234b0bb461450c0b59324a68fb05d029d71cab3a1313b', 'solarseg/engine.py': '9358ebc99df66eb669f32eb83ab8118502f799eaddd9e59d3823b66e6d85e0c5', 'solarseg/gates.py': '4081730062166a57be537f2c4bfebf936b305449a1414356b205b0b1cba753f3', 'solarseg/metrics.py': '9a2d5e5c1bbc7a89250311d1d45af4cda8cfaac8c7eff7cddf5746f69b015b6b', 'solarseg/model.py': '37fe7b0e2edc4997a8221fca12f8eec2b6f5314a665ef89a354a73e9ced1238a', 'solarseg/postprocess.py': '9b779c5079ed33cff64e521612726ee67a5222506aa61738c2bd7d8c861cb9ce', 'solarseg/provenance.py': '2de76b5d199270f50427787e046a1833293697146064b927c342a8d096fa8ad1', 'solarseg/submission.py': '8dc9b08bf2ef1ce473fc28ccc3f49f2fa3e526988cf102d0c4b20d70072ad489', 'requirements.txt': '12fac5b80df1c561b5cd85e4ba29f580ed13ebffd6e024bb3ae26a0339a567d2'}
RUN_TRAINING = False
RUN_HOLDOUT = False
INSTALL_DEPENDENCIES = Path("/kaggle").exists()
REPO = "srivatsav-kannan/solar-seg"

ROOT = Path.cwd()
if not (ROOT / "solarseg").is_dir():
    if (ROOT.parent / "solarseg").is_dir():
        ROOT = ROOT.parent
    else:
        url = f"https://github.com/{REPO}/archive/{SOURCE_COMMIT}.zip"
        payload = urllib.request.urlopen(url, timeout=120).read()
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            for member in archive.infolist():
                target = (ROOT / member.filename).resolve()
                if not target.is_relative_to(ROOT.resolve()):
                    raise ValueError("Unsafe source archive path")
            archive.extractall(ROOT)
        ROOT = ROOT / f"solar-seg-{SOURCE_COMMIT}"
for name, expected in EXPECTED_SOURCE.items():
    actual = hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
    assert actual == expected, f"Source fingerprint differs: {name}"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
assert sys.version_info >= (3, 12), "Use Python 3.12 or newer with the pinned dependencies"
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], check=True)
print("Source revision:", SOURCE_COMMIT)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from solarseg.data import CompetitionData, make_manifest, read_image, sha256
from solarseg.engine import Predictor, train, predict_cache, calibrate, evaluate_and_save
from solarseg.metrics import aggregate, counts
from solarseg.postprocess import encode, decode, instances
from solarseg.provenance import environment, source_hashes, utc_now
from solarseg.submission import write_submission

DATA_ROOT = Path(os.environ.get("SOLARSEG_DATA", "/kaggle/input" if Path("/kaggle/input").exists() else "data/raw"))
WORK = Path("/kaggle/working/solarseg-run") if Path("/kaggle").exists() else ROOT / "artifacts/notebook-run"
WORK.mkdir(parents=True, exist_ok=True)
RELEASE_TAG = 'baseline-v0.1'
MODEL_SHA256 = 'd41c82a2a2310791ff234bc784f765d7b986bf569990ac32a25206c8153f6518'
PARAMS = {'threshold': 0.6, 'min_area': 400, 'closing': 3}
TILE = 0
TTA = True
CHECKPOINT = WORK / "model.pt"
local_model = ROOT / 'artifacts/unet-v2/model.pt'
if not CHECKPOINT.exists():
    if local_model.exists():
        shutil.copy2(local_model, CHECKPOINT)
    else:
        url = f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/model.pt"
        urllib.request.urlretrieve(url, CHECKPOINT)
assert sha256(CHECKPOINT) == MODEL_SHA256, "Checkpoint checksum mismatch"
print(json.dumps(environment(), indent=2))
print("Artifact directory:", WORK)

## 1. Audit inputs and freeze physical-image groups

Download with `python scripts/download_data.py` after accepting the Kaggle rules, or attach
the competition data in Kaggle. An annotation ID is not a physical image ID. All annotators,
crops, and augmentations of one observation follow the same fold. The audit uses 27-day
blocks, exact decoded-pixel hashes, and a three-day embargo on optimization data.

In [ ]:
data = CompetitionData(DATA_ROOT)
audit = make_manifest(data, WORK / "manifests")
assert audit["train_annotation_sha256"] == '5da9e92b5a1a1947fd5d57adb6688269625c48ec1ef884daf2a01618c9ed54a1'
assert audit["split_manifest_sha256"] == 'ebec49b919b111f0591f1740cc10a470c127371203e7522c683c9ec5a34283b8'
manifest = pd.read_csv(WORK / "manifests/train_manifest.csv")
display(pd.Series(audit["roles"], name="Physical images").to_frame())
print("Train images / annotator records / instances / test images:",
      audit["physical_train"], audit["annotator_images"], audit["annotations"], audit["test"])

## 2. Inspect input and annotator-aware target

The foreground target averages separate annotator unions. The score compares predictions
with each annotator separately. It never evaluates against this consensus target.
The example below belongs to calibration, and is selected deterministically.

In [ ]:
stem = manifest.loc[manifest.role == "calibration", "stem"].iloc[0]
image = read_image(data.image_path(stem), 1024)
target = data.soft_target(stem, 1024)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"H-alpha: {stem}")
axes[1].imshow(target, cmap="magma", vmin=0, vmax=1)
axes[1].set_title("Mean of annotator foreground unions")
for ax in axes: ax.axis("off")
plt.show()

## 3. Verify metric and serialization contracts

PQ = sum of matched IoUs / (TP + 0.5 FP + 0.5 FN). An IoU of exactly 0.5 is not a match.
COCO uses compressed counts with column-major ordering, not ordinary Kaggle run-length pairs.
The full repository tests also compare our evaluator with the downloaded official notebook.

In [ ]:
mask = np.zeros((23, 31), dtype=np.uint8)
mask[3:11, 14:25] = 1
np.testing.assert_array_equal(mask, decode(encode(mask)))
assert counts(np.array([[0.5]]))["tp"] == 0
assert aggregate([counts(np.array([[1.0]])), counts(np.zeros((3, 0)))])["pq"] == 0.4
print("Asymmetric COCO roundtrip and pooled PQ contracts passed")

## 4. Train from approved labels (opt-in)

One-channel U-Net; BCE + soft Dice; AdamW; foreground-biased crops mixed with random crops;
rotations/flips and mild intensity/blur augmentation. Training stops after the recorded update
budget. No holdout-based early stopping or external weights are used. Use a new run directory
for each experiment. Training on a different device may produce different learned weights.

In [ ]:
training_run = WORK / "retrained"
if RUN_TRAINING:
    train(DATA_ROOT, WORK / "manifests/train_manifest.csv", training_run,
          steps=6000, size=1024, crop=384,
          batch_size=6, width=24, seed=2026)
    CHECKPOINT = training_run / "model.pt"
else:
    print("Using released checkpoint; set RUN_TRAINING=True to rerun optimization")

## 5. Calibrate instance reconstruction (after optional retraining)

The recorded 25-setting grid includes the initial 16 settings and nine larger-area settings
introduced after inspecting v1 calibration false positives. This is adaptive calibration,
not an unbiased validation estimate. Thresholding follows native-resolution probability
restoration. A small closing operation and area filter precede disjoint connected components.

In [ ]:
predictor = Predictor(CHECKPOINT, device="cpu", tta=TTA, tile=TILE)
if RUN_TRAINING:
    stems = manifest.loc[manifest.role == "calibration", "stem"].tolist()
    cache = training_run / "probabilities-calibration"
    predict_cache(data, stems, predictor, cache)
    selected_result = calibrate(data, stems, cache, training_run)
    PARAMS = selected_result["params"]
else:
    print("Frozen release postprocessing:", PARAMS)

## 6. Evaluate a frozen candidate (opt-in)

The original baseline was frozen before its first holdout evaluation. Those results are now
public and the holdout is no longer secret to the experimenter. Repeating this evaluation
is reproduction, not a fresh independent test. Later model-selection campaigns need nested
grouped CV or a prospectively reserved chronological block; do not tune on this cell's output.

In [ ]:
if RUN_HOLDOUT:
    output = WORK / "holdout-reproduction.json"
    if output.exists():
        raise FileExistsError("Holdout has already been evaluated in this notebook directory")
    stems = manifest.loc[manifest.role == "holdout", "stem"].tolist()
    cache = WORK / "probabilities-holdout"
    predict_cache(data, stems, predictor, cache)
    print(evaluate_and_save(data, stems, cache, PARAMS, output))
else:
    print("Holdout evaluation skipped; measured baseline results are in the repository")

## 7. Infer every test observation and write a valid submission

Test labels are neither available nor required. Every test image is processed, including
images for which the model predicts zero instances. Zero detections go in the coverage
sidecar; the CSV contains only real nonempty masks. IDs preserve the original image stem.
No automatic leaderboard probing, dummy masks, or manual test corrections are performed.

In [ ]:
started = time.monotonic()
stems = sorted(data.test_paths)
cache = WORK / "probabilities-test"
predict_cache(data, stems, predictor, cache)
validation = write_submission(stems, cache, PARAMS, WORK / "submission.csv")
assert validation["images_processed"] == audit["test"]
elapsed = time.monotonic() - started
print(json.dumps(validation, indent=2))
print(f"Inference + serialization seconds (including any cache reuse): {elapsed:.1f}")
display(pd.read_csv(WORK / "submission.csv").head())

## 8. Record evidence; submit only after the gates pass

The CLI gate additionally requires the protected holdout result, reviewed morphology,
official-evaluator parity, source/checkpoint/input fingerprints, and a fresh-kernel notebook
replay. `python scripts/submit.py --run <selected-run> --check-only` checks readiness and quota.
Removing `--check-only` uploads one CSV, records the attempt, and refuses blind duplicate retries.
Check the server's scoring status separately. Maximum: five/day, two final selections.

Final evaluation also requires the public source/checkpoint/notebook, the report in the host's
template, and the separate Google form. A successfully scored CSV does not complete those steps.
The exact form fields require authenticated Google access and were not verified through HTTP.

In [ ]:
proof = {"created_at": utc_now(), "completed": True, "source_commit": SOURCE_COMMIT,
         "source_hashes": source_hashes(), "checkpoint_sha256": sha256(CHECKPOINT),
         "environment": environment(), "validation": validation, "elapsed_seconds": elapsed,
         "training_rerun": RUN_TRAINING, "holdout_rerun": RUN_HOLDOUT}
(WORK / "notebook-proof.json").write_text(json.dumps(proof, indent=2))
if Path("/kaggle").exists():
    shutil.rmtree(cache)  # Generated probability maps are not needed in public outputs.
print("Validated CSV:", WORK / "submission.csv")